# Q1.2 - Extending Faster R-CNN for Oriented Bounding Boxes

## 0. Sanity check - data and GPU

In [1]:
import os, glob, torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

imgs = glob.glob('data/img/*.jpg')
print(f"\nTotal images found: {len(imgs)}")
print("First few:", [os.path.basename(p) for p in imgs[:3]])

CUDA available: True
GPU: NVIDIA GeForce RTX 2080 Ti
VRAM: 11.3 GB

Total images found: 0
First few: []


In [2]:
import os
import sys

cwd = os.path.dirname(os.path.abspath("."))
print(cwd)

sys.path.append(cwd)


/home2/ameyk/cv/assignment-3-ameykaran/q1


In [3]:
import importlib
import train  # or whatever your import is
importlib.reload(train)


<module 'train' from '/home2/ameyk/cv/assignment-3-ameykaran/q1/modified/train.py'>

## 1. Defining the Hyperparameters
We want to try Direct Regression, Multi-bin classification (with various bin sizes), and tune the angle loss weight.

In [4]:
import argparse
import train as train_module

CONFIG = '../config/st.yaml'

# Hyperparameter grids for Oriented Bounding Box prediction
angle_modes = ['regression', 'classification', 'classification']
num_angle_bins_vals = [0, 36, 18, 12] # 0 is placeholder for regression
angle_loss_weights = [1.0, 1.0, 2.0, 2.5]

for i, (mode, bins, w) in enumerate(zip(angle_modes, num_angle_bins_vals, angle_loss_weights)):
    name = f'mode_{mode}_bins_{bins}_w_{w}'
    print(f'Config {i+1}: {name}')

Config 1: mode_regression_bins_0_w_1.0
Config 2: mode_classification_bins_36_w_1.0
Config 3: mode_classification_bins_18_w_2.0


## 2. Quick sanity check - 2 epochs only
Run this cell to verify everything works before committing to the full training run. It temporarily overrides `num_epochs` to 2 and only runs the first configuration.

In [5]:
import yaml

DRY_RUN = False
if DRY_RUN:
    with open(CONFIG) as f:
        cfg = yaml.safe_load(f)
    cfg['train_params']['num_epochs'] = 2

    # Monkey-patch yaml.safe_load to return our modified config
    orig_load = yaml.safe_load
    yaml.safe_load = lambda *a, **kw: cfg

    args = argparse.Namespace(
        config_path=CONFIG,
        rpn_post_nms_top_n_train=2000,
        rpn_fg_iou_thresh=0.8,
        rpn_positive_fraction=0.8,
        angle_mode=angle_modes[0],
        num_angle_bins=num_angle_bins_vals[0],
        angle_loss_weight=angle_loss_weights[0],
    )
    train_module.train(args)

    yaml.safe_load = orig_load  # restore
    print('\nDry run complete!')
else:
    print('DRY_RUN=False, skipping.')

DRY_RUN=False, skipping.


## 3. Full training
Run the training loop for each configuration.

In [ ]:
for mode, bins, w in zip(angle_modes, num_angle_bins_vals, angle_loss_weights):
    name = f'mode_{mode}_bins_{bins}_w_{w}'
    print(f'\n{"="*60}')
    print(f'Starting config: {name}')
    print(f'{"="*60}')
    
    args = argparse.Namespace(
        config_path=CONFIG,
        rpn_post_nms_top_n_train=2000,
        rpn_fg_iou_thresh=0.8,
        rpn_positive_fraction=0.8,
        angle_mode=mode,
        num_angle_bins=bins,
        angle_loss_weight=w,
    )
    
    train_module.train(args)


Starting config: mode_regression_bins_0_w_1.0
{'dataset_params': {'root_dir': '../data', 'num_classes': 2}, 'model_params': {'im_channels': 3, 'aspect_ratios': [0.5, 1, 2], 'scales': [128, 256, 512], 'min_im_size': 600, 'max_im_size': 1000, 'backbone_out_channels': 512, 'fc_inner_dim': 1024, 'rpn_bg_threshold': 0.3, 'rpn_fg_threshold': 0.7, 'rpn_nms_threshold': 0.7, 'rpn_train_prenms_topk': 12000, 'rpn_test_prenms_topk': 6000, 'rpn_train_topk': 2000, 'rpn_test_topk': 300, 'rpn_batch_size': 256, 'rpn_pos_fraction': 0.5, 'roi_iou_threshold': 0.5, 'roi_low_bg_iou': 0.0, 'roi_pool_size': 7, 'roi_nms_threshold': 0.3, 'roi_topk_detections': 100, 'roi_score_threshold': 0.05, 'roi_batch_size': 128, 'roi_pos_fraction': 0.25}, 'train_params': {'task_name': 'st', 'seed': 1111, 'infer_seed': 1122, 'acc_steps': 1, 'num_epochs': 20, 'lr_steps': [12, 16], 'lr': 0.001, 'ckpt_name': 'faster_rcnn_st.pth'}}


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home2/ameyk/.netrc.
wandb: Currently logged in as: ameykaran (iiith-ameykaran) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Val: 100%|██████████| 90/90 [00:11<00:00,  7.93it/s]



===== Epoch 0 =====
  Train Loss     : 8.4632
  Val   Loss     : 10.0186
  OBB-mAP@0.25   : 0.2272
  OBB-mAP@0.50   : 0.0107
  OBB-mAP@0.75   : 0.0000
  Precision@0.50 : 0.0196
  Recall@0.50    : 0.2193
  RPN Cls / Loc  : 0.1956 / 0.1263
  FRCNN Cls / Loc: 0.2891 / 0.3724
  Angle Loss     : 7.4798
